# Module 7: KV Cache, Prefix Caching, Chunked Prefill

## Learning Objectives
- Understand how KV cache speeds decoding
- Implement a simulated KV cache and radix cache
- Benchmark prefill vs. cached prefix
- Measure TTFT improvements
- Connect to Kyro's BlockManager and RadixCache

## Concept: KV Cache

During autoregressive decoding, each token's key and value tensors are cached.
Without KV cache, every new token requires recomputing all previous tokens.
With KV cache, only the new token is computed.

**Kyro mapping**: `src/scheduler/block_manager.rs::BlockManager` and `src/scheduler/radix_cache.rs::RadixCache`.

In [ ]:
import numpy as np
import time

class KVCache:
    def __init__(self, block_size=16, num_blocks=100):
        self.block_size = block_size
        self.num_blocks = num_blocks
        self.cache = {}
        self.free_blocks = list(range(num_blocks))
    def allocate(self, req_id, tokens):
        num_new = (len(tokens) + self.block_size - 1) // self.block_size
        allocated = self.free_blocks[:num_new]
        self.free_blocks = self.free_blocks[num_new:]
        self.cache[req_id] = allocated
        return allocated
    def get(self, req_id):
        return self.cache.get(req_id)

kv = KVCache(block_size=16, num_blocks=100)
prompt = list(range(100))

start = time.time()
blocks = kv.allocate(1, prompt)
prefill_time = time.time() - start

start = time.time()
new_blocks = kv.allocate(2, prompt[80:])
cached_time = time.time() - start

print('=== KV Cache Benchmark ===')
print(f'Full prefill: {prefill_time:.6f}s')
print(f'Cached prefix: {cached_time:.6f}s')
print(f'TTFT improvement: {(1-cached_time/prefill_time)*100:.1f}%')

## Step 2: Radix Cache (Prefix Matching)

Radix caching stores previously computed prefix blocks so that future requests with the same prefix can skip recomputation.

In [ ]:
class RadixCache:
    def __init__(self, capacity=50):
        self.nodes = {}
        self.capacity = capacity
    def insert(self, prefix, blocks):
        self.nodes[tuple(prefix)] = blocks
    def match_prefix(self, tokens):
        for i in range(len(tokens), 0, -1):
            prefix = tuple(tokens[:i])
            if prefix in self.nodes:
                return i, self.nodes[prefix]
        return 0, []

radix = RadixCache(capacity=10)
radix.insert([1, 2, 3, 4, 5], [0, 1])
radix.insert([1, 2, 3, 6, 7], [2, 3])

matched_len, matched_blocks = radix.match_prefix([1, 2, 3, 8, 9])
print(f'Query: [1,2,3,8,9]')
print(f'Matched prefix length: {matched_len}')
print(f'Matched blocks: {matched_blocks}')
print(f'Tokens still needing processing: {5 - matched_len}')

## Exercises
1. Implement a prefix match cache and measure TTFT improvements.
2. Simulate PagedAttention block allocation.
3. Explore `src/scheduler/block_manager.rs::allocate_with_prefix`.